# BLE Test Data Prediction
## Load pre-trained models from models.ipynb and evaluate on test data


### 1. Import Libraries

In [20]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


### 2. Define Feature Engineering Functions

In [21]:
# 5th-floor deployed beacons (order matters: IDs 1..25)
mac_list = [
    'F7:7F:78:76:7E:F3','C6:CD:5E:3D:2F:BB','D6:F4:3A:79:74:63',
    'C9:17:55:E2:3E:0E','CA:60:AB:EE:EC:7F','D6:51:7F:AB:0E:29',
    'CC:54:33:F6:A7:90','EB:20:56:87:04:5A','EE:E7:46:DC:19:6F',
    'C8:5B:BF:37:07:A0','D7:26:F6:A3:44:D2','DD:83:B0:27:FD:36',
    'E5:CD:4A:36:87:06','DC:22:B8:17:4E:B5','EA:09:20:80:D6:44',
    'E6:99:D1:EC:C6:81','F6:DA:97:C7:D5:28','EA:66:A1:12:2C:F4',
    'C9:EA:57:8B:0F:80','D6:7C:1D:2C:2A:0A','DA:E1:70:5F:44:97',
    'DD:10:10:F6:4F:27','E6:F3:93:A8:9E:22','E6:60:05:1F:88:F9',
    'D4:33:FD:F4:C2:A8'
]
beacon_ids = list(range(1, len(mac_list) + 1))
mac_to_id = dict(zip(mac_list, beacon_ids))

def _l2_norm(values: pd.Series) -> float:
    """Calculate L2 norm of RSSI values"""
    arr = values.to_numpy(dtype=float, copy=False)
    return float(np.linalg.norm(arr))

print(f"Beacon list loaded: {len(mac_list)} beacons")

Beacon list loaded: 25 beacons


In [22]:
def make_sliding_window_features(
    df: pd.DataFrame,
    *,
    window_seconds: int = 10,
    step_seconds: int = 5,
    timestamp_col: str = 'timestamp',
    mac_col: str = 'mac address',
    rssi_col: str = 'RSSI',
    include_empty_windows: bool = True,
) -> pd.DataFrame:
    """Feature engineering using sliding window (10s window, 5s hop)"""
    window_td = pd.Timedelta(seconds=window_seconds)
    step_td = pd.Timedelta(seconds=step_seconds)
    
    if window_td <= pd.Timedelta(0) or step_td <= pd.Timedelta(0):
        raise ValueError('window_seconds and step_seconds must be positive')
    if window_td % step_td != pd.Timedelta(0):
        raise ValueError('window_seconds must be a multiple of step_seconds')
    
    required_cols = [timestamp_col, mac_col, rssi_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    x = df[required_cols].copy()
    x[timestamp_col] = pd.to_datetime(x[timestamp_col], errors='coerce')
    x = x.dropna(subset=[timestamp_col])
    x = x.sort_values(timestamp_col, kind='stable')

    if x.empty:
        return pd.DataFrame()
    
    # Convert mac_col to integer beacon_id if it's numeric
    if pd.api.types.is_numeric_dtype(x[mac_col]):
        x['beacon_id'] = x[mac_col].astype(int)
    else:
        # Try to map from mac_to_id
        x['beacon_id'] = x[mac_col].map(mac_to_id)
        # For unmapped MACs, assign NaN (will be excluded)
        x['beacon_id'] = x['beacon_id'].fillna(-1).astype('int64')
        
        # Log unmapped beacons
        unmapped = x[x['beacon_id'] == -1]
        if len(unmapped) > 0:
            unique_unmapped_macs = unmapped[mac_col].unique()
            print(f"WARNING: Found {len(unique_unmapped_macs)} unmapped MAC addresses")
            print(f"  First 5 unmapped: {list(unique_unmapped_macs[:5])}")

    # Each record belongs to the window whose start is floor(timestamp, step),
    # and also the previous overlapping window
    base = x[timestamp_col].dt.floor(step_td)
    n_offsets = int(window_td / step_td)
    parts = []
    for i in range(n_offsets):
        ws = base - (i * step_td)
        temp = x.copy()
        temp['window_start'] = ws
        temp = temp[temp[timestamp_col] < (ws + window_td)]
        parts.append(temp)
    expanded = pd.concat(parts, ignore_index=True) if parts else x.assign(window_start=pd.NaT)

    # Window index
    if include_empty_windows and not expanded.empty:
        win_start = expanded['window_start'].min()
        win_end = expanded['window_start'].max()
        window_index = pd.date_range(start=win_start, end=win_end, freq=step_td, name='window_start')
    else:
        window_index = pd.Index(sorted(expanded['window_start'].unique()), name='window_start')

    out = pd.DataFrame(index=window_index)

    # 1) Per-beacon counts (for known beacons 1..25) - EXCLUDE unmapped (-1)
    known = expanded[expanded['beacon_id'].isin(beacon_ids)].copy()

    counts = (
        known.groupby(['window_start', 'beacon_id'], sort=False)
        .size()
        .unstack(fill_value=0)
        .reindex(index=window_index, columns=beacon_ids, fill_value=0)
        .astype(int)
    )
    counts.columns = [f"count_{i}" for i in counts.columns]
    out = out.join(counts)

    # Unique beacon count (only known beacons)
    out['unique_beacon_count'] = (
        known.groupby('window_start')['beacon_id'].nunique().reindex(window_index).fillna(0).astype(int)
    )

    # 2) Pick b1/b2/b3: top-6 by count, then top-3 by max RSSI
    count_cols = [f"count_{i}" for i in beacon_ids]
    count_arr = out[count_cols].to_numpy(dtype=int, copy=False)

    rssi_max = (
        known.groupby(['window_start', 'beacon_id'])[rssi_col]
        .max()
        .unstack()
        .reindex(index=window_index, columns=beacon_ids)
    )
    rssi_arr = rssi_max.to_numpy(dtype=float, copy=False)
    rssi_arr = np.where(np.isfinite(rssi_arr), rssi_arr, -np.inf)

    n_windows = count_arr.shape[0]
    if n_windows > 0:
        kth = min(5, count_arr.shape[1] - 1)
        top6_unsorted = np.argpartition(-count_arr, kth=kth, axis=1)[:, :6]
        top6_counts = np.take_along_axis(count_arr, top6_unsorted, axis=1)
        order6 = np.argsort(-top6_counts, axis=1, kind='stable')
        top6_idx = np.take_along_axis(top6_unsorted, order6, axis=1)

        cand_counts = np.take_along_axis(count_arr, top6_idx, axis=1)
        cand_rssi = np.take_along_axis(rssi_arr, top6_idx, axis=1)
        cand_rssi = np.where(cand_counts > 0, cand_rssi, -np.inf)
        order3 = np.argsort(-cand_rssi, axis=1, kind='stable')[:, :3]
        top3_idx = np.take_along_axis(top6_idx, order3, axis=1)
        top3_rssi = np.take_along_axis(cand_rssi, order3, axis=1)
        top3_valid = np.isfinite(top3_rssi)
        beacon_id_arr = np.array(beacon_ids, dtype=int)
        top3_ids = beacon_id_arr[top3_idx]
        top3_ids = np.where(top3_valid, top3_ids, 0)
    else:
        top3_idx = np.empty((0, 3), dtype=int)
        top3_valid = np.empty((0, 3), dtype=bool)
        top3_ids = np.empty((0, 3), dtype=int)

    out['b1'] = top3_ids[:, 0] if n_windows else []
    out['b2'] = top3_ids[:, 1] if n_windows else []
    out['b3'] = top3_ids[:, 2] if n_windows else []

    # 3) RSSI stats for b1/b2/b3 - build stats dict for each stat
    stat_names = ['mean', 'std', 'max', 'min', 'norm']
    stats_dict = {}
    
    if not known.empty:
        agg_result = known.groupby(['window_start', 'beacon_id'])[rssi_col].agg(
            mean='mean', std='std', max='max', min='min', norm=_l2_norm
        )
        # agg_result is a DataFrame with multi-index (window_start, beacon_id) and columns (mean, std, max, min, norm)
        for stat_name in stat_names:
            stat_data = agg_result[stat_name].unstack(fill_value=0).reindex(index=window_index, columns=beacon_ids, fill_value=0)
            stats_dict[stat_name] = stat_data.to_numpy(dtype=float, copy=False)

    def _gather(stat_name: str, pos: int) -> np.ndarray:
        if stat_name not in stats_dict or len(stats_dict) == 0:
            return np.zeros(n_windows, dtype=float)
        mat = stats_dict[stat_name]
        mat = np.nan_to_num(mat, nan=0.0, posinf=0.0, neginf=0.0)
        row_idx = np.arange(min(mat.shape[0], n_windows))
        if mat.shape[0] > 0 and mat.shape[1] > 0 and len(top3_idx) > 0:
            col_idx = np.minimum(top3_idx[row_idx, pos], mat.shape[1] - 1)
            values = mat[row_idx, col_idx]
            result = np.zeros(n_windows, dtype=float)
            result[row_idx] = np.where(top3_valid[row_idx, pos], values, 0.0)
            return result
        return np.zeros(n_windows, dtype=float)

    if n_windows:
        out['b1_mean'] = _gather('mean', 0)
        out['b1_std'] = _gather('std', 0)
        out['b1_max'] = _gather('max', 0)
        out['b1_min'] = _gather('min', 0)
        out['b1_norm'] = _gather('norm', 0)

        out['b2_mean'] = _gather('mean', 1)
        out['b2_std'] = _gather('std', 1)
        out['b2_max'] = _gather('max', 1)
        out['b2_min'] = _gather('min', 1)
        out['b2_norm'] = _gather('norm', 1)

        out['b3_mean'] = _gather('mean', 2)
        out['b3_std'] = _gather('std', 2)
        out['b3_max'] = _gather('max', 2)
        out['b3_min'] = _gather('min', 2)
        out['b3_norm'] = _gather('norm', 2)
    else:
        for col in [
            'b1_mean','b1_std','b1_max','b1_min','b1_norm',
            'b2_mean','b2_std','b2_max','b2_min','b2_norm',
            'b3_mean','b3_std','b3_max','b3_min','b3_norm',
        ]:
            out[col] = 0.0

    out = out.sort_index()
    out = out.reset_index()
    stat_cols = [
        'b1_mean','b1_std','b1_max','b1_min','b1_norm',
        'b2_mean','b2_std','b2_max','b2_min','b2_norm',
        'b3_mean','b3_std','b3_max','b3_min','b3_norm',
    ]
    final_cols = ['window_start'] + count_cols + ['b1', 'b2', 'b3'] + stat_cols + ['unique_beacon_count']
    out = out.reindex(columns=final_cols, fill_value=0)

    numeric_cols = [c for c in out.columns if c != 'window_start']
    out[numeric_cols] = out[numeric_cols].fillna(0)

    # Drop windows where no valid beacons were selected
    initial_rows = len(out)
    out = out[~((out['b1'] == 0) & (out['b2'] == 0) & (out['b3'] == 0))]
    dropped_rows = initial_rows - len(out)
    
    if dropped_rows > 0:
        print(f"DROPPED {dropped_rows} windows with no valid beacon detections")

    return out

print("Feature engineering function defined")


Feature engineering function defined


### 3. Load Test Data

In [23]:
# Load test data
test_csv = Path('BLE_Test_predict.csv')
print(f"Loading test data from: {test_csv}")
df_test_raw = pd.read_csv(test_csv, index_col=0)
print(f"Test data shape: {df_test_raw.shape}")
print(f"Columns: {list(df_test_raw.columns)}")
print(f"\nFirst few rows:")
df_test_raw.head()

Loading test data from: BLE_Test_predict.csv
Test data shape: (62222, 5)
Columns: ['user_id', 'timestamp', 'mac address', 'RSSI', 'power']

First few rows:


,user_id,timestamp,mac address,RSSI,power
766,90,2023-04-14 10:01:40,7,-97,-2147483648
764,90,2023-04-14 10:01:40,7,-97,-2147483648
765,90,2023-04-14 10:01:40,7,-97,-2147483648
762,90,2023-04-14 10:01:40,7,-97,-2147483648
761,90,2023-04-14 10:01:40,7,-97,-2147483648


### 4. Apply Feature Engineering on Test Data

In [24]:
print("Applying feature engineering (sliding window 10s/5s step)...")
df_test_fe = make_sliding_window_features(
    df_test_raw,
    window_seconds=10,
    step_seconds=5,
    timestamp_col='timestamp',
    mac_col='mac address',
    rssi_col='RSSI',
    include_empty_windows=False
)
print(f"✓ Engineered feature shape: {df_test_fe.shape}")
print(f"Feature columns: {list(df_test_fe.columns)}")

# DEBUG: Check for rows with all zero beacons
zero_beacon_rows = df_test_fe[((df_test_fe['b1'] == 0) & (df_test_fe['b2'] == 0) & (df_test_fe['b3'] == 0))]
print(f"\nDEBUG: Rows with all zero beacons: {len(zero_beacon_rows)}")
if len(zero_beacon_rows) > 0:
    print("These rows should have been dropped by feature engineering!")
    print(f"Window starts: {zero_beacon_rows['window_start'].head()}")


Applying feature engineering (sliding window 10s/5s step)...
✓ Engineered feature shape: (1861, 45)
Feature columns: ['window_start', 'count_1', 'count_2', 'count_3', 'count_4', 'count_5', 'count_6', 'count_7', 'count_8', 'count_9', 'count_10', 'count_11', 'count_12', 'count_13', 'count_14', 'count_15', 'count_16', 'count_17', 'count_18', 'count_19', 'count_20', 'count_21', 'count_22', 'count_23', 'count_24', 'count_25', 'b1', 'b2', 'b3', 'b1_mean', 'b1_std', 'b1_max', 'b1_min', 'b1_norm', 'b2_mean', 'b2_std', 'b2_max', 'b2_min', 'b2_norm', 'b3_mean', 'b3_std', 'b3_max', 'b3_min', 'b3_norm', 'unique_beacon_count']

DEBUG: Rows with all zero beacons: 0


### 5. Load Pre-trained Models 

In [25]:
print("Checking for pre-trained models...")
print("If not found, we'll auto-generate them from training data.\n")

from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

pipeline_path = Path('trained_pipeline.pkl')
le_path = Path('label_encoder.pkl')

if not pipeline_path.exists() or not le_path.exists():
    print("⚠️  Pre-trained models not found. Auto-generating from training data...")
    
    # Load training data
    print("Loading training data...")
    df_train = pd.read_csv("data/BLE_FE.csv")
    if 'window_start' in df_train.columns:
        df_train = df_train.drop(columns='window_start')
    
    # Prepare X and y
    y_train = df_train['location']
    X_train = df_train.drop(columns='location')
    
    # Encode labels
    print("Training label encoder...")
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    
    # Create and train pipeline
    print("Training XGBoost model...")
    preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), X_train.columns)])
    
    xgb_model = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.02,
        max_depth=5,
        min_child_weight=15,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.5,
        reg_lambda=2.0,
        objective="multi:softprob",
        tree_method="hist",
        random_state=42,
        verbose=0
    )
    
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", xgb_model)
    ])
    
    pipeline.fit(X_train, y_train_encoded)
    
    # Save models
    print("\n✓ Models trained successfully!")
    print(f"Number of classes: {len(le.classes_)}")
    print(f"Classes: {le.classes_}")
    
    joblib.dump(pipeline, pipeline_path)
    joblib.dump(le, le_path)
    print(f"✓ Pipeline exported to: {pipeline_path}")
    print(f"✓ LabelEncoder exported to: {le_path}")
else:
    print("✓ Pre-trained models found!")

Checking for pre-trained models...
If not found, we'll auto-generate them from training data.

✓ Pre-trained models found!


In [26]:
print("Loading pre-trained models...")
pipeline = joblib.load(pipeline_path)
le = joblib.load(le_path)

print(f"✓ Pipeline loaded from: {pipeline_path}")
print(f"✓ LabelEncoder loaded from: {le_path}")
print(f"Classes: {le.classes_}")

Loading pre-trained models...
✓ Pipeline loaded from: trained_pipeline.pkl
✓ LabelEncoder loaded from: label_encoder.pkl
Classes: ['501' '502' '503' '505' '506' '508' '511' '512' '513' '515' '516' '517'
 '518' '520' '522' '523' 'Bathroom' 'cafeteria' 'cleaning' 'hallway'
 'kitchen' 'nurse station']


### 6. Prepare Test Features

In [27]:
# Drop window_start for predictions
X_test = df_test_fe.drop(columns=['window_start'], errors='ignore').copy()
print(f"Test features shape: {X_test.shape}")
print(f"Test features columns: {list(X_test.columns)[:10]}... (showing first 10)")

Test features shape: (1861, 44)
Test features columns: ['count_1', 'count_2', 'count_3', 'count_4', 'count_5', 'count_6', 'count_7', 'count_8', 'count_9', 'count_10']... (showing first 10)


### 7. Make Predictions Using Pre-trained Model

In [28]:
print("Making predictions using pre-trained model...")
y_pred_encoded = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)

# Decode predictions
y_pred = le.inverse_transform(y_pred_encoded)

# DEBUG: Check for unexpected 0 values in predictions
invalid_preds = np.sum(y_pred == 0)
print(f"✓ Predictions completed!")
print(f"Total predictions: {len(y_pred)}")
print(f"Invalid predictions (location = 0): {invalid_preds}")

if invalid_preds > 0:
    print(f"\nWARNING: Found {invalid_preds} predictions with value 0")
    print("This likely means the features contain all zeros (no valid beacons detected)")
    invalid_indices = np.where(y_pred == 0)[0]
    print(f"Indices of invalid predictions: {invalid_indices[:10]}...")

print(f"\nPredicted location distribution:")
print(pd.Series(y_pred).value_counts())


Making predictions using pre-trained model...
✓ Predictions completed!
Total predictions: 1861
Invalid predictions (location = 0): 0

Predicted location distribution:
cafeteria        1218
nurse station     300
hallway           134
kitchen           109
506                51
cleaning           38
515                 9
502                 1
508                 1
Name: count, dtype: int64


### 8. Prepare Output Results

In [29]:
print("Preparing output results...")
results_df = df_test_fe[['window_start']].copy()
results_df['predicted_location'] = y_pred
results_df['confidence'] = y_pred_proba.max(axis=1)

# Add per-class probabilities
for i, class_name in enumerate(le.classes_):
    results_df[f'prob_{class_name}'] = y_pred_proba[:, i]

# Flag problematic predictions (location = 0)
results_df['has_invalid_location'] = (results_df['predicted_location'] == 0)

print(f"Results dataframe shape: {results_df.shape}")
print(f"Invalid predictions (location=0): {results_df['has_invalid_location'].sum()}")

if results_df['has_invalid_location'].sum() > 0:
    print(f"\nDEBUG: Invalid predictions details:")
    invalid_rows = results_df[results_df['has_invalid_location']]
    print(f"  Count: {len(invalid_rows)}")
    print(f"  Average confidence: {invalid_rows['confidence'].mean():.4f}")
    print(f"  First few window_starts with invalid predictions:")
    print(invalid_rows['window_start'].head(10).tolist())

print(f"\nFirst 10 predictions:")
results_df.head(10)


Preparing output results...
Results dataframe shape: (1861, 26)
Invalid predictions (location=0): 0

First 10 predictions:


,window_start,predicted_location,confidence,prob_501,prob_502,prob_503,prob_505,prob_506,prob_508,prob_511,...,prob_520,prob_522,prob_523,prob_Bathroom,prob_cafeteria,prob_cleaning,prob_hallway,prob_kitchen,prob_nurse station,has_invalid_location
0,2023-04-14 10:01:35,nurse station,0.659636,0.001053,0.005153,0.001635,0.001758,0.000882,0.013904,0.005135,...,0.002425,0.001367,0.000515,0.002332,0.144569,0.022545,0.111376,0.004601,0.659636,False
1,2023-04-14 10:01:40,nurse station,0.526856,0.000724,0.000628,0.001142,0.001628,0.000267,0.003090,0.000869,...,0.004218,0.000847,0.000291,0.001977,0.188048,0.024970,0.233394,0.002762,0.526856,False
2,2023-04-14 10:01:45,nurse station,0.470393,0.001405,0.003114,0.002139,0.002108,0.000890,0.002066,0.001593,...,0.005803,0.002903,0.000580,0.002569,0.191254,0.021265,0.276661,0.003797,0.470393,False
3,2023-04-14 10:11:55,cafeteria,0.615028,0.005839,0.014853,0.014257,0.003095,0.001828,0.002500,0.002207,...,0.001853,0.014676,0.180266,0.004404,0.615028,0.012560,0.079311,0.005240,0.014072,False
4,2023-04-14 10:12:00,cafeteria,0.624380,0.003169,0.012708,0.003718,0.002387,0.002287,0.011892,0.005808,...,0.003320,0.019591,0.089110,0.003436,0.624380,0.029900,0.068063,0.002251,0.029208,False
5,2023-04-14 10:12:05,cafeteria,0.654242,0.002123,0.012503,0.003664,0.005411,0.003391,0.013545,0.005498,...,0.003126,0.015310,0.129305,0.003225,0.654242,0.007447,0.061857,0.005424,0.029867,False
6,2023-04-14 10:12:15,cafeteria,0.440867,0.004519,0.004969,0.006537,0.002517,0.004027,0.011637,0.003849,...,0.006640,0.014629,0.017431,0.003665,0.440867,0.202821,0.168647,0.007449,0.074445,False
7,2023-04-14 10:12:20,cafeteria,0.399305,0.001434,0.001005,0.002988,0.001399,0.000599,0.000963,0.003036,...,0.000561,0.002510,0.003063,0.002015,0.399305,0.013949,0.114598,0.006637,0.396185,False
8,2023-04-14 10:12:25,nurse station,0.637127,0.001243,0.002603,0.002127,0.001338,0.006395,0.000867,0.001909,...,0.000507,0.000770,0.000854,0.002084,0.265050,0.036634,0.024349,0.004932,0.637127,False
9,2023-04-14 10:12:30,cafeteria,0.451202,0.000909,0.002906,0.001760,0.001728,0.053815,0.000555,0.002003,...,0.001232,0.000403,0.000410,0.002158,0.451202,0.029219,0.057084,0.005717,0.364748,False


### 9. Save Results to CSV

In [30]:
output_path = Path('BLE_Test_predict_results.csv')
results_df.to_csv(output_path, index=False)
print(f"✓ Results saved to: {output_path.resolve()}")
print(f"\nOutput file info:")
print(f"  Shape: {results_df.shape}")
print(f"  Columns: {list(results_df.columns)}")

✓ Results saved to: C:\Users\umroot\Desktop\BLE Data\scripts\BLE_Test_predict_results.csv

Output file info:
  Shape: (1861, 26)
  Columns: ['window_start', 'predicted_location', 'confidence', 'prob_501', 'prob_502', 'prob_503', 'prob_505', 'prob_506', 'prob_508', 'prob_511', 'prob_512', 'prob_513', 'prob_515', 'prob_516', 'prob_517', 'prob_518', 'prob_520', 'prob_522', 'prob_523', 'prob_Bathroom', 'prob_cafeteria', 'prob_cleaning', 'prob_hallway', 'prob_kitchen', 'prob_nurse station', 'has_invalid_location']


### 10. Final Summary and Metrics

In [31]:
print("\n" + "="*80)
print("PREDICTION RESULTS - TEST DATA EVALUATION")
print("="*80)
print(f"\nRaw test records processed: {df_test_raw.shape[0]:,}")
print(f"Feature-engineered windows: {df_test_fe.shape[0]:,}")
print(f"Predictions generated: {len(y_pred):,}")
print(f"\nModel used: XGBClassifier + Pipeline (from models.ipynb)")
print(f"Number of classes: {len(le.classes_)}")

print(f"\n{'Predicted Location':<20} {'Count':>8} {'Percentage':>12}")
print("-" * 42)
for loc, count in pd.Series(y_pred).value_counts().items():
    pct = 100 * count / len(y_pred)
    print(f"{str(loc):<20} {count:>8} {pct:>11.1f}%")

print(f"\nConfidence Statistics:")
print(f"  Average: {results_df['confidence'].mean():.4f}")
print(f"  Median:  {results_df['confidence'].median():.4f}")
print(f"  Min:     {results_df['confidence'].min():.4f}")
print(f"  Max:     {results_df['confidence'].max():.4f}")
print(f"  Std:     {results_df['confidence'].std():.4f}")

# Count high confidence predictions
high_conf = (results_df['confidence'] > 0.9).sum()
print(f"\nPredictions with confidence > 0.9: {high_conf} ({100*high_conf/len(results_df):.1f}%)")

print(f"\n✓ Results saved to: BLE_Test_predict_results.csv")
print("="*80)


PREDICTION RESULTS - TEST DATA EVALUATION

Raw test records processed: 62,222
Feature-engineered windows: 1,861
Predictions generated: 1,861

Model used: XGBClassifier + Pipeline (from models.ipynb)
Number of classes: 22

Predicted Location      Count   Percentage
------------------------------------------
cafeteria                1218        65.4%
nurse station             300        16.1%
hallway                   134         7.2%
kitchen                   109         5.9%
506                        51         2.7%
cleaning                   38         2.0%
515                         9         0.5%
502                         1         0.1%
508                         1         0.1%

Confidence Statistics:
  Average: 0.5795
  Median:  0.5792
  Min:     0.1981
  Max:     0.9599
  Std:     0.1407

Predictions with confidence > 0.9: 11 (0.6%)

✓ Results saved to: BLE_Test_predict_results.csv
